In [2]:
# ============================================================
# FILE 7 - AWARENESS INSIGHTS & VISUALIZATION
# CORRECTED DATASET PATH VERSION
# ============================================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ------------------------------------------------------------
# 1. PROJECT PATH
# ------------------------------------------------------------

PROJECT_PATH = "/content/drive/MyDrive/CEP Seasonal Disease Prediction"

print("=" * 70)
print("FILE 7 - AWARENESS INSIGHTS & VISUALIZATION")
print("=" * 70)

print("\nProject Path:")
print(PROJECT_PATH)

# ------------------------------------------------------------
# 2. CHECK PROJECT PATH
# ------------------------------------------------------------

if not os.path.exists(PROJECT_PATH):

    raise FileNotFoundError(
        f"""
Project folder not found:

{PROJECT_PATH}

Please make sure Google Drive is mounted and the
'CEP Seasonal Disease Prediction' folder exists.
"""
    )

# ------------------------------------------------------------
# 3. SEARCH FOR DATASET AUTOMATICALLY
# ------------------------------------------------------------

TARGET_FILE = "Maharashtra_Feature_Engineered.csv"

FOUND_FILES = []

for root, dirs, files in os.walk(PROJECT_PATH):

    if TARGET_FILE in files:

        FOUND_FILES.append(
            os.path.join(root, TARGET_FILE)
        )

print("\nSearching for feature-engineered dataset...")

if len(FOUND_FILES) == 0:

    print("\nNo feature-engineered dataset found.")

    print("\nFiles currently inside processed folder:")

    PROCESSED_PATH = os.path.join(
        PROJECT_PATH,
        "Data",
        "processed"
    )

    if os.path.exists(PROCESSED_PATH):

        for root, dirs, files in os.walk(PROCESSED_PATH):

            for file in files:

                print(
                    os.path.join(
                        root,
                        file
                    )
                )

    raise FileNotFoundError(
        """
Maharashtra_Feature_Engineered.csv was not found
anywhere inside the project folder.

Please verify that File 5 successfully created the
feature-engineered dataset.
"""
    )

# ------------------------------------------------------------
# 4. DISPLAY FOUND DATASETS
# ------------------------------------------------------------

print("\nFeature-engineered files found:")

for file in FOUND_FILES:

    print("✓", file)

# ------------------------------------------------------------
# 5. SELECT DATASET
# ------------------------------------------------------------

# Prefer Feature_Engineering folder if available
preferred_files = [
    file
    for file in FOUND_FILES
    if "Feature_Engineering" in file
]

if len(preferred_files) > 0:

    DATA_PATH = preferred_files[0]

else:

    DATA_PATH = FOUND_FILES[0]

print("\nSelected Dataset:")
print(DATA_PATH)

print("\nFile Exists:")
print(os.path.exists(DATA_PATH))

# ------------------------------------------------------------
# 6. OUTPUT DIRECTORIES
# ------------------------------------------------------------

OUTPUT_PATH = os.path.join(
    PROJECT_PATH,
    "Data",
    "processed",
    "Awareness_Insights"
)

FIGURE_PATH = os.path.join(
    OUTPUT_PATH,
    "Figures"
)

TABLE_PATH = os.path.join(
    OUTPUT_PATH,
    "Tables"
)

os.makedirs(
    OUTPUT_PATH,
    exist_ok=True
)

os.makedirs(
    FIGURE_PATH,
    exist_ok=True
)

os.makedirs(
    TABLE_PATH,
    exist_ok=True
)

# ------------------------------------------------------------
# 7. LOAD DATA
# ------------------------------------------------------------

df = pd.read_csv(DATA_PATH)

print("\nDataset Loaded Successfully")

print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

# ------------------------------------------------------------
# 8. STANDARDIZE COLUMNS
# ------------------------------------------------------------

df.columns = df.columns.str.strip()

# Date
if "Date" in df.columns:

    df["Date"] = pd.to_datetime(
        df["Date"],
        errors="coerce"
    )

elif all(
    col in df.columns
    for col in ["Year", "Month", "Day"]
):

    df["Date"] = pd.to_datetime(
        dict(
            year=df["Year"],
            month=df["Month"],
            day=df["Day"]
        ),
        errors="coerce"
    )

else:

    raise ValueError(
        "No usable Date information found."
    )

# District
if "District" not in df.columns:

    if "district" in df.columns:

        df["District"] = df["district"]

    else:

        print(
            "\nWarning: District column not found."
        )

# Required columns
required_columns = [
    "Date",
    "Disease",
    "Cases"
]

missing_columns = [
    col
    for col in required_columns
    if col not in df.columns
]

if missing_columns:

    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

# ------------------------------------------------------------
# 9. NUMERIC CONVERSION
# ------------------------------------------------------------

numeric_columns = [
    "Cases",
    "Deaths",
    "Temp",
    "preci",
    "LAI"
]

for col in numeric_columns:

    if col in df.columns:

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

# ------------------------------------------------------------
# 10. BASIC CLEANING
# ------------------------------------------------------------

df = df.dropna(
    subset=[
        "Date",
        "Disease",
        "Cases"
    ]
).copy()

df = df[
    df["Cases"] >= 0
].copy()

# ------------------------------------------------------------
# 11. TIME FEATURES
# ------------------------------------------------------------

df["Year"] = df["Date"].dt.year

df["Month"] = df["Date"].dt.month

df["Month_Name"] = (
    df["Date"]
    .dt.month_name()
)

df["Month_Short"] = (
    df["Date"]
    .dt.strftime("%b")
)

# ------------------------------------------------------------
# 12. SEASON
# ------------------------------------------------------------

def assign_season(month):

    if month in [3, 4, 5]:

        return "Summer"

    elif month in [6, 7, 8, 9]:

        return "Monsoon"

    elif month in [10, 11]:

        return "Post-Monsoon"

    else:

        return "Winter"


df["Season"] = df["Month"].apply(
    assign_season
)

season_order = [
    "Summer",
    "Monsoon",
    "Post-Monsoon",
    "Winter"
]

month_order = [
    "Jan",
    "Feb",
    "Mar",
    "Apr",
    "May",
    "Jun",
    "Jul",
    "Aug",
    "Sep",
    "Oct",
    "Nov",
    "Dec"
]

df["Season"] = pd.Categorical(
    df["Season"],
    categories=season_order,
    ordered=True
)

# ------------------------------------------------------------
# 13. VISUAL SETTINGS
# ------------------------------------------------------------

sns.set_theme(
    style="whitegrid",
    context="talk"
)

plt.rcParams["figure.dpi"] = 150

plt.rcParams["savefig.dpi"] = 300

plt.rcParams["axes.titleweight"] = "bold"

plt.rcParams["axes.titlesize"] = 18

plt.rcParams["axes.labelsize"] = 12

print("\nData preparation completed.")

# ============================================================
# 14. DATASET OVERVIEW
# ============================================================

district_count = (
    df["District"].nunique()
    if "District" in df.columns
    else np.nan
)

dataset_overview = pd.DataFrame({

    "Metric": [

        "Total Records",

        "Total Reported Cases",

        "Number of Diseases",

        "Number of Districts",

        "Number of Years",

        "Start Year",

        "End Year"

    ],

    "Value": [

        len(df),

        int(df["Cases"].sum()),

        df["Disease"].nunique(),

        district_count,

        df["Year"].nunique(),

        int(df["Year"].min()),

        int(df["Year"].max())

    ]

})

dataset_overview.to_csv(

    os.path.join(
        TABLE_PATH,
        "dataset_overview.csv"
    ),

    index=False
)

print("\nDataset Overview:")

display(dataset_overview)

# ============================================================
# 15. DISEASE SUMMARY
# ============================================================

disease_summary = (

    df.groupby(
        "Disease",
        as_index=False
    )

    .agg(

        Total_Cases=(
            "Cases",
            "sum"
        ),

        Average_Cases=(
            "Cases",
            "mean"
        ),

        Maximum_Cases=(
            "Cases",
            "max"
        ),

        Records=(
            "Cases",
            "count"
        )

    )

    .sort_values(
        "Total_Cases",
        ascending=False
    )
)

disease_summary[
    "Case_Share_Percent"
] = (

    disease_summary[
        "Total_Cases"
    ]

    /

    disease_summary[
        "Total_Cases"
    ].sum()

    * 100

)

disease_summary.to_csv(

    os.path.join(
        TABLE_PATH,
        "disease_summary.csv"
    ),

    index=False
)

display(
    disease_summary
)

# ============================================================
# 16. GRAPH - DISEASE RANKING
# ============================================================

plot_data = (
    disease_summary
    .sort_values(
        "Total_Cases"
    )
)

plt.figure(
    figsize=(15, 10)
)

sns.barplot(
    data=plot_data,
    x="Total_Cases",
    y="Disease"
)

plt.title(
    "Reported Disease Cases in Maharashtra",
    pad=20
)

plt.xlabel(
    "Total Reported Cases"
)

plt.ylabel(
    "Disease"
)

plt.tight_layout()

plt.savefig(

    os.path.join(
        FIGURE_PATH,
        "01_total_cases_by_disease.png"
    ),

    bbox_inches="tight"
)

plt.show()

plt.close()

# ============================================================
# 17. YEARLY ANALYSIS
# ============================================================

yearly_summary = (

    df.groupby(
        "Year",
        as_index=False
    )

    .agg(
        Total_Cases=(
            "Cases",
            "sum"
        )
    )

)

yearly_summary.to_csv(

    os.path.join(
        TABLE_PATH,
        "yearly_summary.csv"
    ),

    index=False
)

plt.figure(
    figsize=(16, 8)
)

sns.lineplot(

    data=yearly_summary,

    x="Year",

    y="Total_Cases",

    marker="o",

    linewidth=3

)

plt.title(
    "Year-wise Trend of Reported Disease Cases",
    pad=20
)

plt.xlabel("Year")

plt.ylabel(
    "Reported Cases"
)

plt.xticks(
    yearly_summary["Year"],
    rotation=45
)

plt.tight_layout()

plt.savefig(

    os.path.join(
        FIGURE_PATH,
        "02_yearly_disease_trend.png"
    ),

    bbox_inches="tight"
)

plt.show()

plt.close()

# ============================================================
# 18. MONTHLY ANALYSIS
# ============================================================

monthly_summary = (

    df.groupby(
        [
            "Month",
            "Month_Short"
        ],
        as_index=False
    )

    .agg(
        Total_Cases=(
            "Cases",
            "sum"
        )
    )

    .sort_values(
        "Month"
    )

)

monthly_summary.to_csv(

    os.path.join(
        TABLE_PATH,
        "monthly_summary.csv"
    ),

    index=False
)

plt.figure(
    figsize=(16, 8)
)

sns.lineplot(

    data=monthly_summary,

    x="Month_Short",

    y="Total_Cases",

    marker="o",

    linewidth=3

)

plt.title(
    "Monthly Seasonal Pattern of Reported Cases",
    pad=20
)

plt.xlabel("Month")

plt.ylabel(
    "Reported Cases"
)

plt.tight_layout()

plt.savefig(

    os.path.join(
        FIGURE_PATH,
        "03_monthly_seasonal_pattern.png"
    ),

    bbox_inches="tight"
)

plt.show()

plt.close()

# ============================================================
# 19. SEASONAL ANALYSIS
# ============================================================

seasonal_summary = (

    df.groupby(
        "Season",
        observed=True,
        as_index=False
    )

    .agg(

        Total_Cases=(
            "Cases",
            "sum"
        ),

        Average_Cases=(
            "Cases",
            "mean"
        ),

        Maximum_Cases=(
            "Cases",
            "max"
        ),

        Records=(
            "Cases",
            "count"
        )

    )

)

seasonal_summary.to_csv(

    os.path.join(
        TABLE_PATH,
        "seasonal_summary.csv"
    ),

    index=False
)

plt.figure(
    figsize=(13, 8)
)

sns.barplot(

    data=seasonal_summary,

    x="Season",

    y="Total_Cases",

    order=season_order

)

plt.title(
    "Reported Disease Cases by Season",
    pad=20
)

plt.xlabel("Season")

plt.ylabel(
    "Total Reported Cases"
)

plt.tight_layout()

plt.savefig(

    os.path.join(
        FIGURE_PATH,
        "04_seasonal_cases.png"
    ),

    bbox_inches="tight"
)

plt.show()

plt.close()

# ============================================================
# 20. DISEASE × SEASON HEATMAP
# ============================================================

disease_season = pd.pivot_table(

    df,

    values="Cases",

    index="Disease",

    columns="Season",

    aggfunc="sum",

    fill_value=0

)

disease_season = disease_season.reindex(
    columns=season_order
)

disease_season.to_csv(

    os.path.join(
        TABLE_PATH,
        "disease_season_summary.csv"
    )
)

plt.figure(
    figsize=(15, 10)
)

sns.heatmap(

    disease_season,

    annot=True,

    fmt=".0f",

    cmap="YlOrRd",

    linewidths=0.5

)

plt.title(
    "Disease-wise Seasonal Distribution",
    pad=20
)

plt.xlabel("Season")

plt.ylabel("Disease")

plt.tight_layout()

plt.savefig(

    os.path.join(
        FIGURE_PATH,
        "05_disease_season_heatmap.png"
    ),

    bbox_inches="tight"
)

plt.show()

plt.close()

# ============================================================
# 21. DISEASE × MONTH HEATMAP
# ============================================================

disease_month = pd.pivot_table(

    df,

    values="Cases",

    index="Disease",

    columns="Month_Short",

    aggfunc="sum",

    fill_value=0

)

disease_month = disease_month.reindex(
    columns=month_order
)

disease_month.to_csv(

    os.path.join(
        TABLE_PATH,
        "disease_month_summary.csv"
    )
)

plt.figure(
    figsize=(17, 10)
)

sns.heatmap(

    disease_month,

    annot=True,

    fmt=".0f",

    cmap="YlGnBu",

    linewidths=0.4

)

plt.title(
    "Disease-wise Monthly Distribution",
    pad=20
)

plt.xlabel("Month")

plt.ylabel("Disease")

plt.tight_layout()

plt.savefig(

    os.path.join(
        FIGURE_PATH,
        "06_disease_month_heatmap.png"
    ),

    bbox_inches="tight"
)

plt.show()

plt.close()

# ============================================================
# 22. DISTRICT ANALYSIS
# ============================================================

if "District" in df.columns:

    district_summary = (

        df.groupby(
            "District",
            as_index=False
        )

        .agg(

            Total_Cases=(
                "Cases",
                "sum"
            ),

            Average_Cases=(
                "Cases",
                "mean"
            ),

            Maximum_Cases=(
                "Cases",
                "max"
            ),

            Records=(
                "Cases",
                "count"
            )

        )

        .sort_values(
            "Total_Cases",
            ascending=False
        )

    )

    district_summary[
        "Case_Share_Percent"
    ] = (

        district_summary[
            "Total_Cases"
        ]

        /

        district_summary[
            "Total_Cases"
        ].sum()

        * 100

    )

    district_summary.to_csv(

        os.path.join(
            TABLE_PATH,
            "district_summary.csv"
        ),

        index=False
    )

    # TOP 15
    top15 = (
        district_summary
        .head(15)
        .sort_values(
            "Total_Cases"
        )
    )

    plt.figure(
        figsize=(15, 10)
    )

    sns.barplot(

        data=top15,

        x="Total_Cases",

        y="District"

    )

    plt.title(
        "Top 15 Maharashtra Districts by Reported Cases",
        pad=20
    )

    plt.xlabel(
        "Total Reported Cases"
    )

    plt.ylabel(
        "District"
    )

    plt.tight_layout()

    plt.savefig(

        os.path.join(
            FIGURE_PATH,
            "07_top_15_districts.png"
        ),

        bbox_inches="tight"
    )

    plt.show()

    plt.close()

# ============================================================
# 23. DISTRICT × DISEASE HEATMAP
# ============================================================

if "District" in df.columns:

    district_disease = pd.pivot_table(

        df,

        values="Cases",

        index="District",

        columns="Disease",

        aggfunc="sum",

        fill_value=0

    )

    district_disease.to_csv(

        os.path.join(
            TABLE_PATH,
            "district_disease_summary.csv"
        )
    )

    top20 = (
        district_summary
        .head(20)["District"]
        .tolist()
    )

    heatmap_data = district_disease.loc[
        district_disease.index.isin(top20)
    ]

    heatmap_data = heatmap_data.loc[
        heatmap_data.sum(
            axis=1
        ).sort_values(
            ascending=False
        ).index
    ]

    plt.figure(
        figsize=(18, 11)
    )

    sns.heatmap(

        heatmap_data,

        cmap="OrRd",

        linewidths=0.3

    )

    plt.title(
        "Disease Distribution Across Top Maharashtra Districts",
        pad=20
    )

    plt.xlabel("Disease")

    plt.ylabel("District")

    plt.xticks(
        rotation=45,
        ha="right"
    )

    plt.tight_layout()

    plt.savefig(

        os.path.join(
            FIGURE_PATH,
            "08_district_disease_heatmap.png"
        ),

        bbox_inches="tight"
    )

    plt.show()

    plt.close()

# ============================================================
# 24. PEAK MONTH
# ============================================================

peak_month_temp = (

    df.groupby(
        [
            "Disease",
            "Month_Short"
        ],
        as_index=False
    )

    .agg(
        Total_Cases=(
            "Cases",
            "sum"
        )
    )

)

peak_month = (

    peak_month_temp

    .sort_values(
        [
            "Disease",
            "Total_Cases"
        ],

        ascending=[
            True,
            False
        ]
    )

    .groupby(
        "Disease"
    )

    .first()

    .reset_index()

)

peak_month = peak_month.rename(

    columns={
        "Month_Short": "Peak_Month",
        "Total_Cases": "Peak_Month_Cases"
    }

)

peak_month.to_csv(

    os.path.join(
        TABLE_PATH,
        "peak_month_by_disease.csv"
    ),

    index=False
)

# ============================================================
# 25. PEAK SEASON
# ============================================================

peak_season_temp = (

    df.groupby(
        [
            "Disease",
            "Season"
        ],
        observed=True,
        as_index=False
    )

    .agg(
        Total_Cases=(
            "Cases",
            "sum"
        )
    )

)

peak_season = (

    peak_season_temp

    .sort_values(
        [
            "Disease",
            "Total_Cases"
        ],

        ascending=[
            True,
            False
        ]
    )

    .groupby(
        "Disease"
    )

    .first()

    .reset_index()

)

peak_season = peak_season.rename(

    columns={
        "Season": "Peak_Season",
        "Total_Cases": "Peak_Season_Cases"
    }

)

peak_season.to_csv(

    os.path.join(
        TABLE_PATH,
        "peak_season_by_disease.csv"
    ),

    index=False
)

# ============================================================
# 26. CLIMATE RELATIONSHIP
# ============================================================

climate_results = []

climate_variables = {

    "Temperature": "Temp",

    "Precipitation": "preci",

    "LAI": "LAI"

}

for name, column in climate_variables.items():

    if column not in df.columns:

        continue

    temp_data = df[
        [
            "Cases",
            column
        ]
    ].dropna()

    if len(temp_data) < 3:

        continue

    correlation = temp_data[
        "Cases"
    ].corr(
        temp_data[column],
        method="spearman"
    )

    climate_results.append({

        "Variable": name,

        "Column": column,

        "Spearman_Correlation": correlation,

        "Observations": len(temp_data)

    })

climate_relationship_summary = pd.DataFrame(
    climate_results
)

climate_relationship_summary.to_csv(

    os.path.join(
        TABLE_PATH,
        "climate_relationship_summary.csv"
    ),

    index=False
)

print("\nClimate Relationship Analysis:")

display(
    climate_relationship_summary
)

# ============================================================
# 27. TEMPERATURE VS CASES
# ============================================================

if "Temp" in df.columns:

    temp_data = df[
        [
            "Temp",
            "Cases"
        ]
    ].dropna()

    plt.figure(
        figsize=(12, 8)
    )

    sns.scatterplot(

        data=temp_data,

        x="Temp",

        y="Cases",

        alpha=0.55

    )

    plt.title(
        "Temperature and Reported Disease Cases",
        pad=20
    )

    plt.xlabel(
        "Temperature"
    )

    plt.ylabel(
        "Reported Cases"
    )

    plt.tight_layout()

    plt.savefig(

        os.path.join(
            FIGURE_PATH,
            "09_temperature_vs_cases.png"
        ),

        bbox_inches="tight"
    )

    plt.show()

    plt.close()

# ============================================================
# 28. PRECIPITATION VS CASES
# ============================================================

if "preci" in df.columns:

    precip_data = df[
        [
            "preci",
            "Cases"
        ]
    ].dropna()

    plt.figure(
        figsize=(12, 8)
    )

    sns.scatterplot(

        data=precip_data,

        x="preci",

        y="Cases",

        alpha=0.55

    )

    plt.title(
        "Precipitation and Reported Disease Cases",
        pad=20
    )

    plt.xlabel(
        "Precipitation"
    )

    plt.ylabel(
        "Reported Cases"
    )

    plt.tight_layout()

    plt.savefig(

        os.path.join(
            FIGURE_PATH,
            "10_precipitation_vs_cases.png"
        ),

        bbox_inches="tight"
    )

    plt.show()

    plt.close()

# ============================================================
# 29. LAI VS CASES
# ============================================================

if "LAI" in df.columns:

    lai_data = df[
        [
            "LAI",
            "Cases"
        ]
    ].dropna()

    plt.figure(
        figsize=(12, 8)
    )

    sns.scatterplot(

        data=lai_data,

        x="LAI",

        y="Cases",

        alpha=0.55

    )

    plt.title(
        "Vegetation Index (LAI) and Reported Disease Cases",
        pad=20
    )

    plt.xlabel(
        "LAI"
    )

    plt.ylabel(
        "Reported Cases"
    )

    plt.tight_layout()

    plt.savefig(

        os.path.join(
            FIGURE_PATH,
            "11_lai_vs_cases.png"
        ),

        bbox_inches="tight"
    )

    plt.show()

    plt.close()

# ============================================================
# 30. YEAR × SEASON HEATMAP
# ============================================================

year_season = pd.pivot_table(

    df,

    values="Cases",

    index="Year",

    columns="Season",

    aggfunc="sum",

    fill_value=0

)

year_season = year_season.reindex(
    columns=season_order
)

year_season.to_csv(

    os.path.join(
        TABLE_PATH,
        "year_season_summary.csv"
    )
)

plt.figure(
    figsize=(14, 10)
)

sns.heatmap(

    year_season,

    annot=True,

    fmt=".0f",

    cmap="YlOrRd",

    linewidths=0.3

)

plt.title(
    "Reported Disease Cases by Year and Season",
    pad=20
)

plt.xlabel("Season")

plt.ylabel("Year")

plt.tight_layout()

plt.savefig(

    os.path.join(
        FIGURE_PATH,
        "12_year_season_heatmap.png"
    ),

    bbox_inches="tight"
)

plt.show()

plt.close()

# ============================================================
# 31. DISEASE CASE SHARE
# ============================================================

top_diseases = (
    disease_summary
    .head(8)
    .copy()
)

plt.figure(
    figsize=(14, 9)
)

sns.barplot(

    data=top_diseases.sort_values(
        "Case_Share_Percent"
    ),

    x="Case_Share_Percent",

    y="Disease"

)

plt.title(
    "Share of Reported Cases by Major Disease Categories",
    pad=20
)

plt.xlabel(
    "Share of Reported Cases (%)"
)

plt.ylabel(
    "Disease"
)

plt.tight_layout()

plt.savefig(

    os.path.join(
        FIGURE_PATH,
        "13_disease_case_share.png"
    ),

    bbox_inches="tight"
)

plt.show()

plt.close()

# ============================================================
# 32. SEASONAL DISTRIBUTION
# ============================================================

plt.figure(
    figsize=(13, 8)
)

sns.boxplot(

    data=df,

    x="Season",

    y="Cases",

    order=season_order

)

plt.title(
    "Distribution of Reported Cases Across Seasons",
    pad=20
)

plt.xlabel("Season")

plt.ylabel(
    "Reported Cases"
)

plt.tight_layout()

plt.savefig(

    os.path.join(
        FIGURE_PATH,
        "14_seasonal_case_distribution.png"
    ),

    bbox_inches="tight"
)

plt.show()

plt.close()

# ============================================================
# 33. AUTOMATIC INSIGHTS
# ============================================================

insights = []

# Disease
if len(disease_summary) > 0:

    row = disease_summary.iloc[0]

    insights.append({

        "Category": "Disease",

        "Insight":
        f"{row['Disease']} has the highest total "
        f"reported case volume in the analyzed dataset."

    })

# Season
if len(seasonal_summary) > 0:

    row = seasonal_summary.sort_values(
        "Total_Cases",
        ascending=False
    ).iloc[0]

    insights.append({

        "Category": "Season",

        "Insight":
        f"{row['Season']} has the highest total "
        f"reported case volume among the analyzed seasons."

    })

# Month
if len(monthly_summary) > 0:

    row = monthly_summary.sort_values(
        "Total_Cases",
        ascending=False
    ).iloc[0]

    insights.append({

        "Category": "Month",

        "Insight":
        f"{row['Month_Short']} has the highest total "
        f"reported case volume among the analyzed months."

    })

# District
if (
    "District" in df.columns
    and len(district_summary) > 0
):

    row = district_summary.iloc[0]

    insights.append({

        "Category": "District",

        "Insight":
        f"{row['District']} has the highest "
        f"reported case volume in the analyzed dataset."

    })

# Disease peak months
for _, row in peak_month.iterrows():

    insights.append({

        "Category": "Disease Peak Month",

        "Insight":
        f"{row['Disease']} shows its highest "
        f"reported case volume in {row['Peak_Month']}."

    })

# Disease peak seasons
for _, row in peak_season.iterrows():

    insights.append({

        "Category": "Disease Peak Season",

        "Insight":
        f"{row['Disease']} shows its highest "
        f"reported case volume in the "
        f"{row['Peak_Season']} season."

    })

awareness_insights = pd.DataFrame(
    insights
)

awareness_insights.to_csv(

    os.path.join(
        TABLE_PATH,
        "awareness_insights.csv"
    ),

    index=False
)

# ============================================================
# 34. WEBSITE SUMMARY
# ============================================================

website_summary = pd.DataFrame({

    "Metric": [

        "Total Records",

        "Total Reported Cases",

        "Diseases",

        "Districts",

        "Start Year",

        "End Year"

    ],

    "Value": [

        len(df),

        int(df["Cases"].sum()),

        df["Disease"].nunique(),

        df["District"].nunique()
        if "District" in df.columns
        else np.nan,

        df["Year"].min(),

        df["Year"].max()

    ]

})

website_summary.to_csv(

    os.path.join(
        TABLE_PATH,
        "website_summary.csv"
    ),

    index=False
)

# ============================================================
# 35. FINAL OUTPUT
# ============================================================

print("\n")
print("=" * 70)
print("FILE 7 COMPLETED SUCCESSFULLY")
print("=" * 70)

print("\nDataset Used:")
print(DATA_PATH)

print("\nFigures saved to:")
print(FIGURE_PATH)

print("\nTables saved to:")
print(TABLE_PATH)

print("\nNumber of figures:")
print(
    len(
        os.listdir(
            FIGURE_PATH
        )
    )
)

print("\nNumber of tables:")
print(
    len(
        os.listdir(
            TABLE_PATH
        )
    )
)

print("\nGenerated Figures:")

for file in sorted(
    os.listdir(FIGURE_PATH)
):

    print("✓", file)

print("\nGenerated Tables:")

for file in sorted(
    os.listdir(TABLE_PATH)
):

    print("✓", file)

print("\n")
print("=" * 70)
print("AWARENESS ANALYSIS READY")
print("=" * 70)

print(
    """
IMPORTANT INTERPRETATION:

These results describe patterns in the analyzed
dataset.

They should NOT be interpreted as:

- Medical diagnosis
- Treatment advice
- Proof of disease causation
- Individual health risk prediction

The website will use these results for
community awareness and educational purposes.
"""
)

FILE 7 - AWARENESS INSIGHTS & VISUALIZATION

Project Path:
/content/drive/MyDrive/CEP Seasonal Disease Prediction

Searching for feature-engineered dataset...

No feature-engineered dataset found.

Files currently inside processed folder:


FileNotFoundError: 
Maharashtra_Feature_Engineered.csv was not found
anywhere inside the project folder.

Please verify that File 5 successfully created the
feature-engineered dataset.


In [3]:
import os

PROJECT_PATH = "/content/drive/MyDrive/CEP Seasonal Disease Prediction"

print("=" * 70)
print("CHECKING CEP PROJECT STRUCTURE")
print("=" * 70)

print("\nProject exists:")
print(os.path.exists(PROJECT_PATH))

print("\nProject path:")
print(PROJECT_PATH)

print("\nProject contents:")

if os.path.exists(PROJECT_PATH):

    for item in os.listdir(PROJECT_PATH):

        print("📁/📄", item)

        item_path = os.path.join(
            PROJECT_PATH,
            item
        )

        if os.path.isdir(item_path):

            print("   └── Folder contents:")

            for subitem in os.listdir(item_path):

                print("       ", subitem)

else:

    print("❌ Project folder not found.")

print("\n" + "=" * 70)
print("SEARCHING ENTIRE MY DRIVE")
print("=" * 70)

MY_DRIVE = "/content/drive/MyDrive"

matches = []

for root, dirs, files in os.walk(MY_DRIVE):

    for file in files:

        if (
            "Maharashtra" in file
            or
            "Feature" in file
            or
            "Seasonal" in file
        ):

            matches.append(
                os.path.join(
                    root,
                    file
                )
            )

print("\nPossible matching files:")

if matches:

    for file in matches:

        print("✓", file)

else:

    print(
        "❌ No Maharashtra/Feature/Seasonal files found."
    )

print("\n" + "=" * 70)

CHECKING CEP PROJECT STRUCTURE

Project exists:
True

Project path:
/content/drive/MyDrive/CEP Seasonal Disease Prediction

Project contents:
📁/📄 Data
   └── Folder contents:
        processed

SEARCHING ENTIRE MY DRIVE

Possible matching files:
❌ No Maharashtra/Feature/Seasonal files found.



In [4]:
import os

PROJECT_PATH = "/content/drive/MyDrive/CEP Seasonal Disease Prediction"

print("=" * 70)
print("CHECKING COMPLETE CEP PROJECT")
print("=" * 70)

for root, dirs, files in os.walk(PROJECT_PATH):

    level = root.replace(PROJECT_PATH, "").count(os.sep)

    indent = "    " * level

    print(f"{indent}📁 {os.path.basename(root)}/")

    for file in files:

        print(f"{indent}    📄 {file}")

print("\n" + "=" * 70)

# Specifically check raw folder
RAW_PATH = os.path.join(
    PROJECT_PATH,
    "Data",
    "raw"
)

print("RAW FOLDER:")
print(RAW_PATH)

print("\nRaw folder exists:")
print(os.path.exists(RAW_PATH))

if os.path.exists(RAW_PATH):

    raw_files = os.listdir(RAW_PATH)

    if raw_files:

        for file in raw_files:
            print("✓", file)

    else:

        print("❌ Raw folder is empty.")

CHECKING COMPLETE CEP PROJECT
📁 CEP Seasonal Disease Prediction/
    📁 Data/
        📁 processed/
            📁 Awareness_Insights/
                📁 Figures/
                📁 Tables/

RAW FOLDER:
/content/drive/MyDrive/CEP Seasonal Disease Prediction/Data/raw

Raw folder exists:
False
